Steps:
1. Import packages needed
2. Input data (includes train test split, normalize)
3. Create model with Hyperparameter analysis
4. Acquire parameters / Evaluate model
5. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)

## 1. Import Packages

In [299]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [300]:
np.set_printoptions(precision=5)
random_state = 42
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.model_selection import validation_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV

## 2. Input Data

In [301]:
mtgjson_data_df = pd.read_csv('data/finalCards.csv')
mtgjson_data_df['text'] = mtgjson_data_df['text'].astype('string').fillna('')

creature_dummies_before = [
    'isAlternative',
    'isGameChanger', 
    'isPromo',
    'isReprint', 
    'isReserved',
    'isHuman', 
    'isElemental',
    'isDragon', 
    'isSpirit', 
    'isAngel', 
    'isElf', 
    'isVampire', 
    'isZombie',
    'isBeast', 
    'isWizard', 
    'isSoldier', 
    'isKnight', 
    'isCleric', 
    'isWarrior',
    'isRogue', 
    'isShaman', 
    'isDruid', 
    'isCreature', 
    'isPlaneswalker', 
    'isMTGO', 
    'isFlying',
    'isLegal',
    'isBanned']

for col in creature_dummies_before:
    mtgjson_data_df[col] = mtgjson_data_df[col].map({True: 1, False: 0})

creatures_df = mtgjson_data_df[mtgjson_data_df['isCreature'] == 1]
creatures_df.head()
creatures_df = creatures_df[['uuid', 
                            'cardName', 
                            'availability', 
                            'colorIdentity', 
                            'edhrecRank', 
                            'edhrecSaltiness', 
                            'finishes', 
                            'isAlternative', 
                            'isGameChanger',
                            'isPromo',
                            'isReprint',
                            'isReserved',
                            'manaValue',
                            'cardNumber',
                            'power',
                            'rarity',
                            'setCode',
                            'subtypes',
                            'text',
                            'toughness',
                            'types',
                            'price',
                            'commander',
                            'setName',
                            'releaseDate',
                            'scryfallId',
                            'isHuman', 
                            'isElemental',
                            'isDragon', 
                            'isSpirit', 
                            'isAngel', 
                            'isElf',
                            'isVampire', 
                            'isZombie',
                            'isBeast', 
                            'isWizard',
                            'isEldrazi', 
                            'isSoldier', 
                            'isKnight', 
                            'isCleric', 
                            'isWarrior',
                            'isRogue', 
                            'isShaman', 
                            'isDruid', 
                            'isCreature', 
                            'isMTGO', 
                            'isFlying',
                            'isLegal',
                            'isBanned'
                            ]]

one_hot_encoded = pd.get_dummies(creatures_df['rarity'], prefix="rarity:")
creatures_df = pd.concat([creatures_df,one_hot_encoded], axis=1)
creatures_df = creatures_df.drop('rarity', axis=1)

print(len(creatures_df))
creatures_df = creatures_df.dropna()
print(len(creatures_df))


44836
22218


### Add Lemmatization Step for cleaning text

In [302]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# download libraries
nltk.download('wordnet')
nltk.download('punkt')

def lemmatize_sentence(sentence):
    lemmatizer = WordNetLemmatizer()
    #Should words be lemmatized? (Have to lemmatize words based on noun and verb...)
    tokens = word_tokenize(sentence)
    lemmatized_tokens = [lemmatizer.lemmatize(token)+' ' for token in tokens]
    return ''.join(lemmatized_tokens)

#lemmatize mtgjson avility text
creatures_df['text_lemmatized'] = creatures_df['text'].apply(lemmatize_sentence)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Count specific Lemmatized Text
#### Acquire the top 100 frequent words one-hot encode them

In [303]:
#Acquire counts for each word (not including stop words)
min_ngram = 1
max_ngram = 3
vectorizer = CountVectorizer(max_features=5, stop_words='english', ngram_range=(min_ngram, max_ngram))
model = vectorizer.fit_transform(creatures_df['text_lemmatized'])
ability_text = vectorizer.get_feature_names_out()
stop_words_vect = vectorizer.get_stop_words()
columns_text = ["contains: "+ item for item in ability_text]
words_df = pd.DataFrame(model.toarray(), columns=columns_text)
words_df.head(10)
final_df = creatures_df.join(words_df).fillna(0)

### Train Test Split MTGJSON Data

In [304]:
X_columns = [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank',
        'edhrecSaltiness', 
        'manaValue',
        'isReprint',
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard',
        'isEldrazi',
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        'isCreature', 
        'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'rarity:_common',
        'rarity:_mythic',
        'rarity:_rare',
        'rarity:_special',
        'rarity:_uncommon'
        ] + columns_text
y_column = "price"
X_columns.remove(y_column)

X = final_df[X_columns]
y = final_df[y_column]

#Split Data
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, random_state=random_state)

#Normalize data for non binary columns
scaled_columns = [
        'power',
        'toughness',
        'edhrecRank',
        'edhrecSaltiness',
        'manaValue']
X_train_norm= X_train_raw.copy()
X_test_norm = X_test_raw.copy()
train_features = X_train_norm[scaled_columns]
test_features = X_test_norm[scaled_columns]

scaler = StandardScaler().fit(train_features.values)
train_features = scaler.transform(train_features.values)
test_features = scaler.transform(test_features.values)

X_train_norm[scaled_columns] = train_features
X_test_norm[scaled_columns] = test_features


## 3. Create Model with Hyperparameter Tuning

In [305]:
#Create Ridge Regression Model
ridge_clf = Ridge()

#Create Lasso Regression Model
#lasso_clf = Lasso()

#Parameters with and GridSearch
parameters = {
    'alpha' : [0.25, 0.5, 0.75, 1]
    }

ridge_clf_hyp = GridSearchCV(
    estimator=ridge_clf,
    param_grid=parameters,
    cv=5,
)

ridge_clf_hyp.fit(X_train_norm, y_train_raw)

#lasso_clf_hyp = GridSearchCV(
#    estimator=lasso_clf,
#    param_grid=parameters,
#    cv=5
#)

#lasso_clf_hyp.fit(X_train_norm, y_train_raw)

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.25, 0.5, 0.75, 1]})

## 4. Acquire Parameters / Evaluate Metrics

In [306]:
ridge_parameters = ridge_clf_hyp.get_params
#lasso_parameters = lasso_clf_hyp.get_params
ridge_y_predict = ridge_clf_hyp.predict(X_test_norm)
#lasso_y_predict = lasso_clf_hyp.predict(X_test_norm)
ridge_r2_score = ridge_clf_hyp.score(X_test_norm, y_test_raw)
#lasso_r2_score = lasso_clf_hyp.score(X_test_norm, y_test_raw)
ridge_mse = mean_squared_error(y_test_raw, ridge_y_predict)
#lasso_mse = mean_squared_error(y_test_raw, lasso_y_predict)
best_model_ridge = ridge_clf_hyp.best_estimator_
#best_model_lasso = lasso_clf_hyp.best_estimator_
print("RIDGE MODEL")
print(f"Best Model: {best_model_ridge}")
print(f"Model Coefficients: {best_model_ridge.coef_}")
print(f"R2 Score: {ridge_r2_score}")
print(f"Mean Square error: {ridge_mse}")
print("----------------------------------------")
#print("LASSO MODEL")
#print(f"Best Model: {best_model_lasso}")
#print(f"Model Coefficients: {best_model_lasso.coef_}")
#print(f"R2 Score: {lasso_r2_score}")
#print(f"Mean Square error: {lasso_mse}")


#Evaluate feature_importance
feature_importance = pd.DataFrame({
    'Feature': X_columns,
    'Coefficients': best_model_ridge.coef_,
})

feature_importance['Abs_Val_Coefficients'] = abs(feature_importance['Coefficients'])
feature_importance.sort_values('Abs_Val_Coefficients', ascending=False, inplace=True)
print(feature_importance.head(15))

RIDGE MODEL
Best Model: Ridge(alpha=1)
Model Coefficients: [  0.33333   0.16146  -0.86232   5.64036  -1.75755  -4.66652  -1.46755
   1.55804   3.93244  -1.67477  -2.16166  -2.48712  -3.14326  -4.00352
  -2.77803  -0.36143  -4.5165   -1.18059   2.16343  -5.32603  -2.9934
  -5.96171  -4.16818  -2.75271   0.       -9.30854   1.44716 -18.51097
   0.       -6.80866   3.88649  -2.30907  11.25088  -6.01964   2.73834
   0.20471  -1.26136   1.15038   0.05514]
R2 Score: 0.018980040409102394
Mean Square error: 8674.906186737275
----------------------------------------
             Feature  Coefficients  Abs_Val_Coefficients
27           isLegal    -18.510971             18.510971
32   rarity:_special     11.250880             11.250880
25            isMTGO     -9.308539              9.308539
29    rarity:_common     -6.808655              6.808655
33  rarity:_uncommon     -6.019643              6.019643
21           isRogue     -5.961712              5.961712
3    edhrecSaltiness      5.640365   

## 5. Visualizations